# Attention Mechanisms — Expert
> Section 01 — Topic 03 — expert

**Prerequisites:** 03-attention-mechanisms/advanced.ipynb

**What you'll learn:**
- Attention sinks and why first tokens get disproportionate attention
- How to extract and analyze attention patterns from real models
- Sparse and linear attention for ultra-long sequences

**What you'll build:**
- An attention analysis toolkit for real pretrained models

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. Attention Sinks — Why First Tokens Get Disproportionate Attention

A striking pattern observed in trained language models is that the **first few tokens** in a sequence consistently receive high attention weights from all subsequent tokens, regardless of their content. This phenomenon, called "attention sinks" (Xiao et al., 2023), occurs because softmax attention must sum to 1.0 — when a token doesn't need to attend strongly anywhere, it dumps excess attention weight onto the first tokens as a kind of "no-op" sink.

This has practical implications. **StreamingLLM** leverages attention sinks to enable infinite-length generation: by always keeping the first few tokens (the "sink tokens") plus a sliding window of recent tokens in the KV cache, the model maintains stable generation quality indefinitely. Without the sink tokens, perplexity degrades rapidly as the window slides past the beginning of the sequence.

The attention sink phenomenon also reveals something about how transformers learn to distribute attention. The first tokens serve as a learned "default" or "rest position" for attention heads that don't have strong content-based signals for a particular query.

In [ ]:
# Load a small model and extract attention patterns
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_attentions=True)
model.eval()

# Encode a passage
text = "The transformer architecture has revolutionized natural language processing. Since its introduction in 2017, it has become the backbone of nearly every state-of-the-art model in the field."
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions is a tuple of (batch, heads, seq_q, seq_k) per layer
print(f"Model: {model_name}")
print(f"Layers: {len(outputs.attentions)}")
print(f"Heads per layer: {outputs.attentions[0].shape[1]}")
print(f"Sequence length: {outputs.attentions[0].shape[2]}")
print(f"Tokens: {tokens[:10]}...")

In [ ]:
# Visualize attention sinks: average attention to each position
n_layers = len(outputs.attentions)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for layer_idx, ax in enumerate(axes.flat):
    if layer_idx >= n_layers:
        ax.axis('off')
        continue
    
    # Average attention received by each key position (across all queries and heads)
    attn = outputs.attentions[layer_idx][0]  # (heads, seq_q, seq_k)
    avg_received = attn.mean(dim=(0, 1))  # Average over heads and queries
    
    ax.bar(range(len(avg_received)), avg_received.numpy(), color='steelblue', alpha=0.7)
    ax.set_title(f'Layer {layer_idx}', fontsize=10)
    ax.set_xlim(-1, len(avg_received))
    if layer_idx % 4 == 0:
        ax.set_ylabel('Avg attention received')

fig.suptitle('Attention Sinks: Average Attention Received by Each Token Position', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("Notice: the first token (position 0) consistently receives high attention across layers.")
print("This is the 'attention sink' phenomenon.")

## 2. Attention Pattern Analysis — Extract and Visualize from Real Models

Analyzing attention patterns reveals what different layers and heads have learned. Research has identified several common patterns: **positional heads** (attending to nearby tokens), **induction heads** (copying patterns from earlier in the sequence), **syntactic heads** (tracking grammatical structure), and **rare word heads** (attending to unusual or informative tokens).

Understanding these patterns is valuable for model interpretability, debugging, and architecture design. For example, the discovery of induction heads — pairs of attention heads that work together to implement in-context learning — was a breakthrough in understanding how transformers learn from their context.

In [ ]:
# Classify attention head patterns
def analyze_head_patterns(attention_weights, tokens):
    """Analyze attention patterns for each head in a layer."""
    n_heads, seq_len, _ = attention_weights.shape
    patterns = []
    
    for head in range(n_heads):
        attn = attention_weights[head].numpy()  # (seq_q, seq_k)
        
        # 1. Diagonal score: how much does each token attend to the previous token?
        diag_score = np.mean([attn[i, i-1] for i in range(1, seq_len)])
        
        # 2. First-token score: how much attention goes to position 0?
        sink_score = np.mean([attn[i, 0] for i in range(1, seq_len)])
        
        # 3. Entropy: how spread out is the attention?
        entropy = -np.sum(attn * np.log(attn + 1e-10), axis=-1).mean()
        
        # Classify
        if diag_score > 0.3:
            pattern = "previous-token"
        elif sink_score > 0.3:
            pattern = "attention-sink"
        elif entropy > 2.5:
            pattern = "distributed"
        else:
            pattern = "sparse/specific"
        
        patterns.append({
            'head': head,
            'pattern': pattern,
            'diag_score': diag_score,
            'sink_score': sink_score,
            'entropy': entropy
        })
    
    return patterns

# Analyze patterns across all layers
print(f"{'Layer':<6} {'Head':<6} {'Pattern':<18} {'Prev-tok':<10} {'Sink':<10} {'Entropy':<10}")
print("-" * 60)

for layer_idx in [0, 3, 6, 11]:  # Sample layers
    attn = outputs.attentions[layer_idx][0]
    patterns = analyze_head_patterns(attn, tokens)
    for p in patterns[:4]:  # First 4 heads
        print(f"{layer_idx:<6} {p['head']:<6} {p['pattern']:<18} {p['diag_score']:<10.3f} {p['sink_score']:<10.3f} {p['entropy']:<10.3f}")

In [ ]:
# Visualize specific interesting heads
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Pick interesting layers/heads to show different patterns
examples = [(0, 0), (0, 5), (3, 0), (6, 3), (9, 7), (11, 0)]
display_tokens = tokens[:20]  # Show first 20 tokens for readability

for (layer, head), ax in zip(examples, axes.flat):
    attn = outputs.attentions[layer][0, head, :20, :20].numpy()
    im = ax.imshow(attn, cmap='Blues', vmin=0)
    ax.set_title(f'Layer {layer}, Head {head}', fontsize=10)
    
    if len(display_tokens) <= 20:
        ax.set_xticks(range(len(display_tokens)))
        ax.set_xticklabels(display_tokens, rotation=90, fontsize=6)
        ax.set_yticks(range(len(display_tokens)))
        ax.set_yticklabels(display_tokens, fontsize=6)

fig.suptitle('Attention Patterns from GPT-2 (diverse heads)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Sparse Attention — BigBird, Longformer Patterns

For very long sequences (thousands to millions of tokens), even Flash Attention's $O(N)$ memory doesn't help with the $O(N^2)$ computational cost. Sparse attention methods reduce this by only computing attention for a subset of position pairs.

**Longformer** (Beltagy et al., 2020) combines three patterns: (1) **local attention** — a sliding window around each token, (2) **dilated attention** — windows with gaps to capture patterns at different scales, and (3) **global attention** — special tokens (like [CLS]) that attend to and are attended by all tokens.

**BigBird** (Zaheer et al., 2020) proved that sparse attention with local + global + random connections is a universal approximator — it can theoretically compute anything that full attention can. The random connections are surprisingly important: they create shortcuts in the attention graph that enable information to flow between distant tokens efficiently.

In [ ]:
def create_sparse_attention_mask(seq_len, window_size=3, n_global=2, n_random=2):
    """Create a combined local + global + random sparse attention mask."""
    mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
    
    # 1. Local attention: sliding window
    for i in range(seq_len):
        start = max(0, i - window_size)
        end = min(seq_len, i + window_size + 1)
        mask[i, start:end] = True
    
    # 2. Global attention: first n_global tokens attend to/from everything
    mask[:n_global, :] = True  # Global tokens attend to all
    mask[:, :n_global] = True  # All tokens attend to global tokens
    
    # 3. Random attention: each token attends to n_random random positions
    for i in range(seq_len):
        random_positions = torch.randperm(seq_len)[:n_random]
        mask[i, random_positions] = True
    
    # Apply causal constraint
    causal = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
    mask = mask & causal
    
    return mask

# Visualize sparse attention patterns
seq_len = 32
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Full causal
full_mask = torch.tril(torch.ones(seq_len, seq_len))
axes[0].imshow(full_mask.numpy(), cmap='Greens')
axes[0].set_title(f'Full Causal\n({int(full_mask.sum())} connections)')

# Local only
local_mask = create_sparse_attention_mask(seq_len, window_size=3, n_global=0, n_random=0)
axes[1].imshow(local_mask.float().numpy(), cmap='Greens')
axes[1].set_title(f'Local Only (w=3)\n({int(local_mask.sum())} connections)')

# Local + Global
lg_mask = create_sparse_attention_mask(seq_len, window_size=3, n_global=2, n_random=0)
axes[2].imshow(lg_mask.float().numpy(), cmap='Greens')
axes[2].set_title(f'Local + Global\n({int(lg_mask.sum())} connections)')

# BigBird-style: Local + Global + Random
bigbird_mask = create_sparse_attention_mask(seq_len, window_size=3, n_global=2, n_random=3)
axes[3].imshow(bigbird_mask.float().numpy(), cmap='Greens')
axes[3].set_title(f'BigBird (L+G+R)\n({int(bigbird_mask.sum())} connections)')

for ax in axes:
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.tight_layout()
plt.show()
print(f"Sparsity reduction: {int(full_mask.sum())} -> {int(bigbird_mask.sum())} = {bigbird_mask.sum()/full_mask.sum()*100:.0f}% of full")

## 4. Linear Attention — Kernel-Based Approximations

Linear attention methods replace the softmax attention mechanism with kernel-based approximations that reduce the complexity from $O(N^2)$ to $O(N)$ in both time and memory. The key insight is that if we can decompose the attention kernel $K(q, k) = \phi(q)^T \phi(k)$ for some feature map $\phi$, we can rearrange the computation.

Standard attention: $\text{Attn} = \text{softmax}(QK^T)V$ requires computing the $N \times N$ matrix $QK^T$ first.

Linear attention: $\text{Attn} = \phi(Q)(\phi(K)^T V)$ — by computing $\phi(K)^T V$ first (a $d \times d$ matrix), then multiplying by $\phi(Q)$, we avoid the $N \times N$ bottleneck entirely.

The tradeoff is approximation quality. Various feature maps have been proposed — random Fourier features (Performers), ELU-based features, and others. In practice, linear attention works well for some tasks but struggles with the sharp, peaked attention patterns that softmax naturally produces.

In [ ]:
def standard_softmax_attention(Q, K, V):
    """Standard O(N^2) softmax attention."""
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V)

def linear_attention_elu(Q, K, V):
    """Linear attention with ELU feature map. O(N*d^2) instead of O(N^2*d)."""
    # Feature map: elu(x) + 1 (ensures non-negative)
    Q_prime = F.elu(Q) + 1  # (batch, seq, d)
    K_prime = F.elu(K) + 1  # (batch, seq, d)
    
    # Compute K^T V first: (batch, d, d) — this is the key trick
    KV = torch.matmul(K_prime.transpose(-2, -1), V)  # (batch, d, d)
    
    # Then Q * (K^T V): (batch, seq, d)
    numerator = torch.matmul(Q_prime, KV)
    
    # Normalization: sum of kernel values per query
    denominator = torch.matmul(Q_prime, K_prime.sum(dim=-2, keepdim=True).transpose(-2, -1))
    
    return numerator / (denominator + 1e-6)

def linear_attention_random_features(Q, K, V, n_features=64):
    """Linear attention with random Fourier features (Performer-style)."""
    d_k = Q.shape[-1]
    
    # Random projection matrix
    omega = torch.randn(d_k, n_features) / math.sqrt(d_k)
    
    # Approximate softmax kernel with random features
    # phi(x) = exp(-||x||^2/2) * [cos(omega^T x), sin(omega^T x)] / sqrt(m)
    def feature_map(x):
        proj = torch.matmul(x, omega)  # (batch, seq, n_features)
        return torch.cat([torch.cos(proj), torch.sin(proj)], dim=-1) / math.sqrt(n_features)
    
    Q_prime = feature_map(Q)
    K_prime = feature_map(K)
    
    KV = torch.matmul(K_prime.transpose(-2, -1), V)
    numerator = torch.matmul(Q_prime, KV)
    denominator = torch.matmul(Q_prime, K_prime.sum(dim=-2, keepdim=True).transpose(-2, -1))
    
    return numerator / (denominator + 1e-6)

# Compare outputs
seq_len, d_k = 32, 64
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

out_standard = standard_softmax_attention(Q, K, V)
out_elu = linear_attention_elu(Q, K, V)
out_rff = linear_attention_random_features(Q, K, V, n_features=128)

print(f"Cosine similarity to standard attention:")
print(f"  ELU linear: {F.cosine_similarity(out_standard.flatten(), out_elu.flatten(), dim=0).item():.4f}")
print(f"  RFF linear: {F.cosine_similarity(out_standard.flatten(), out_rff.flatten(), dim=0).item():.4f}")

## 5. Attention in Practice — What Production Models Use

Understanding which attention techniques are used in real production models helps guide architecture decisions. Here's the current landscape.

In [ ]:
# Summary table of attention techniques in production models
models = {
    "GPT-4 (speculated)": {"KV": "MHA or GQA", "Position": "Unknown", "Efficiency": "Flash Attention", "Context": "128K"},
    "Llama 3 8B": {"KV": "GQA (8 KV heads)", "Position": "RoPE", "Efficiency": "Flash Attention 2", "Context": "8K (128K extended)"},
    "Llama 3 70B": {"KV": "GQA (8 KV heads)", "Position": "RoPE", "Efficiency": "Flash Attention 2", "Context": "8K (128K extended)"},
    "Mistral 7B": {"KV": "GQA (8 KV heads)", "Position": "RoPE", "Efficiency": "Sliding Window + Flash", "Context": "32K"},
    "Gemma 2 27B": {"KV": "GQA", "Position": "RoPE", "Efficiency": "Sliding + Global layers", "Context": "8K"},
    "Phi-3": {"KV": "GQA", "Position": "RoPE (LongRoPE)", "Efficiency": "Flash Attention", "Context": "128K"},
    "Claude 3.5": {"KV": "Unknown", "Position": "Unknown", "Efficiency": "Unknown", "Context": "200K"},
}

print(f"{'Model':<22} {'KV Strategy':<20} {'Position':<16} {'Efficiency':<24} {'Context':<16}")
print("=" * 100)
for name, info in models.items():
    print(f"{name:<22} {info['KV']:<20} {info['Position']:<16} {info['Efficiency']:<24} {info['Context']:<16}")

print("\n Key trends:")
print("  - GQA has won over MHA and MQA for the KV strategy")
print("  - RoPE is the dominant positional encoding")
print("  - Flash Attention is universal — every model uses it")
print("  - Context lengths are growing rapidly: 8K -> 128K -> 200K+")

## Summary

**Key takeaways:**
- **Attention sinks** at the first token positions are a universal phenomenon — understanding this enables StreamingLLM for infinite generation
- **Attention pattern analysis** reveals head specialization: positional, syntactic, semantic, and induction heads
- **Sparse attention** (local + global + random) achieves O(N) complexity with provable expressiveness (BigBird)
- **Linear attention** replaces softmax with kernel approximations for O(N) time and memory, but with quality tradeoffs
- Production models combine multiple techniques: GQA + RoPE + Flash Attention is the current standard

**Next:** [03-attention-mechanisms/build.ipynb](build.ipynb) — Implement Flash Attention and benchmark